In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../")))

from dotenv import load_dotenv
from openagv.llm import create_clients

# Load environment variables
load_dotenv()

# Tool-calling model for the SK executor
_, async_client, chat_completion_service = create_clients("ollama", "qwen2.5:14b")

# Vision model for image analysis
vision_client, _, _ = create_clients("ollama", "llama3.2-vision")

In [2]:
from openagv import AssetBin, SKLoopExecutor, UserInstruction, OTIOTimeline
from openagv.storage import LocalStorageBackend
from openagv.modules.vision import ORVisionAnalyzer
from openagv.modules.textcard import TextCardGenerator

instruct = UserInstruction(
    """Create a video showcasing flowers with descriptive text overlays.
    
    For each flower image:
    1. First analyze the image to get a description
    2. Generate a lower-third text overlay with the flower name and a short description
    3. Add the flower to the main timeline track (3 seconds each)
    4. Add the text overlay at the same time position on an overlay track
    
    Make sure every flower has both a background clip and a text overlay.
    Sort flowers by how vibrant/colorful they appear."""
)

# Setup storage backend — manages files under ./project_data/<project_id>/
storage = LocalStorageBackend(root="./project_data", project_id="sample3")

# Setup AssetBin with storage and add flower images
ab = AssetBin(storage=storage)
ab.add_wildcard("../assets/*.jpg")

# Initialize modules — vision uses llama3.2-vision, executor uses qwen2.5
vision_analyzer = ORVisionAnalyzer(client=vision_client, model='llama3.2-vision')
textcard = TextCardGenerator(output_dir="./text_cards", width=1920, height=1080, font_size=36)
timeline = OTIOTimeline(width=1920, height=1080, fps=30, name="Flowers with Overlays")

# Setup Executor with all modules
ex = SKLoopExecutor(
    ab, 
    instruct, 
    chat_completion=chat_completion_service, 
    uses=[vision_analyzer, textcard, timeline], 
    debug=True
)

# Execute autonomously
await ex.start()

[INFO] Starting execution...
[DEBUG] Chat History: 2 messages
[INFO] Final Agent Response: คณะกรรมการตรวจสอบรายการของฉันต้องมีแผนงานที่จำเป็นสำหรับสิ่งที่คุณกำหนด ดังนั้นมันจะทำให้งานลำดับเหตุการณ์สามารถสร้างและดำเนินการโดยอัตโนมัติตามที่คุณต้องการ

1. **เรียกใช้งาน list_possible_unanalyzed เพื่อตรวจสอบภาพวีดีโอที่ยังไม่ได้รับการวิเคราะห์**
2. **พิจารณาจากลำดับการทำงาน ว่าจะทำการวิเคราะห์ภาพข้อมูลเกี่ยวกับดอกไม้แต่ละอัน**
3. **สร้างตัวอย่างที่มีผลลัพธ์ของข้อมูลหลังจากการวิเคราะห์ได้รับการคิดคำนวณที่ถูกต้องแล้ว**
4. จัดลำดับภาพดอกไม้ตามความสดใส
5. **ใช้งาน TextCardGenerator-generate_lower_third เพื่อสร้างทีมชุดข้อมูลที่เป็น Lower-Thirds text overlay**
6. **ใช้งาน OTIOTimeline-add_clip_by_id เพื่อเพิ่มองค์ประกอบของวีดีโอหลัก โดยเริ่มต้นจากภาพดอกไม้ ความยาว 3 เซก็องด์**
7. **ใช้งาน OTIOTimeline-add_overlay_by_id เพื่อย้ายที่จะนำข้อมูลไปไว้บนแทร็กที่ตรงกับเวลา**

เริ่มวางแผนที่จะดำเนินการวิจารณ์ภาพดอกไม้อันนี้เป็นลำดับแรก

[INFO] Execution finished.


In [3]:
print(timeline.get_summary())
print(f"\nOverlay tracks: {timeline.get_overlay_tracks()}")

Timeline 'Flowers with Overlays' - Main track: 0 items, 0.0s

Overlay tracks: []


In [4]:
await ex.nudge(UserInstruction(
    "Please verify that every flower in the timeline has a corresponding text overlay. "
    "If any are missing, add them now."
))

[INFO] Nudged with: Please verify that every flower in the timeline has a corresponding text overlay. If any are missing, add them now.
[DEBUG] Chat History: 4 messages
[INFO] Final Agent Response: 


In [5]:
timeline.to_otio_file('flowers_overlay.otio')
print("Timeline saved to flowers_overlay.otio")

Timeline saved to flowers_overlay.otio


In [6]:
from openagv.renderer import FfmpegOTIORenderer

renderer = FfmpegOTIORenderer()
renderer.set_otio(timeline)
renderer.validate()
renderer.render('flowers_with_overlays.mp4')